In [4]:
import os
from typing import List
from pydantic import BaseModel
from langchain_openai import OpenAIEmbeddings,ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import StateGraph,END
from langchain_core.documents import Document
from langchain_community.document_loaders import TextLoader

docs = TextLoader("internaldocs.txt").load()
splitter = RecursiveCharacterTextSplitter(chunk_size = 500,chunk_overlap=50)
chunks = splitter.split_documents(documents=docs)
embedding = OpenAIEmbeddings()
vs = FAISS.from_documents(documents=chunks,embedding=OpenAIEmbeddings())
retriever = vs.as_retriever()

llm = ChatOpenAI(model="gpt-4o-mini")

class RAGCotState(BaseModel):
    question : str
    sub_steps : List[str] = []
    retriver_docs : List[Document] = []
    answer : str = ""

def plan_steps(state:RAGCotState)->RAGCotState:
    prompt = f"Break the question into 2-3 reasoning steps \n\n {state.question}"
    result = llm.invoke(prompt)
    sub_qs = [line.strip("- ") for line in result.split("\n") if line.strip()]

    return state.model_copy(update={"sub_steps":sub_qs})

def retriever_per_step(state:RAGCotState)->RAGCotState:
    all_docs = []
    for sub in state.sub_steps:
        doc = retriever.invoke(sub)
        all_docs.extend(doc)

    return state.model_copy({"retriver_docs" : all_docs})



In [ ]:
def generate_answer(state:RAGCotState)->RAGCotState:
    
    context = "\n\n".join([doc.page_content for doc in state.retriver_docs])
    prompt = f"""You are answering a complex question by using reasoning and reteirved docs

    Question : {state.question}

    Relevant Information : {context}

    Now synthesize a well reasoned answer."""

    result = llm.invoke(prompt).content.strip()
    return state.model_copy(update={"answer":result})

